In [3]:
# ============================================================================
# CELL 1: Install dependencies
# ============================================================================
#@title Install dependencies
#@markdown This installs RDKit and the PostgreSQL database driver (takes a few sec)

import os
import sys
from IPython.display import Image, display

# Use the raw content URL
url = "https://raw.githubusercontent.com/AzizAbusaleh/PCCL/main/pccl_rxns_a1-a8.png"

print("Installing RDKit and PostgreSQL drivers...")
!pip install -q rdkit pandas psycopg2-binary

print("✓ Installation complete!")
# Display the image
display(Image(url=url))

Installing RDKit and PostgreSQL drivers...
✓ Installation complete!


## 🟢 Cell 2 — Engine + GUI
Run this cell to load everything and show the GUI. The cell defines all the database/decomposition logic and then displays the interface.

In [13]:
# ============================================================================
# CELL 2: Clean Separate-Table Cloud Engine (FINAL VERSION)
# ============================================================================
#@title Load Cloud Engine & Hit Expansion Logic

import psycopg2
import pandas as pd
import itertools
import time
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs, Descriptors, Lipinski, Crippen, QED
from rdkit.Chem import rdFingerprintGenerator

# Suppress RDKit's verbose C++ logger output.
# These warnings ("atom N in product 0 has multiple H count specifications")
# come from the reverse-decomposition step in RunReactants. We handle them
# correctly via _normalize_reagent below, so the messages are just noise.
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")


class PCCLDatabase:
    def __init__(self):
        self.conn_params = {
            "host": "206.12.98.193",
            "database": "pccl_db2",
            "user": "pccl_guest2",
            "password": "guest_readonly_2026"
        }
        # Direct table mapping - no JOINs needed!
        self.table_map = {
            "Rousseaux_arthorBBs_OH1": "rousseaux_oh1",
            "Rousseaux_arthorBBs_Br1": "rousseaux_br1",
            "Rousseaux_arthorBBs_C001": "rousseaux_coo1",
            "Rousseaux_arthorBBs_amines": "rousseaux_amines",
            "Le_arthorBBs_amines": "le_amines",
            "Beauchemin_arthorBBs_alkynes": "beauchemin_alkynes",
            "Beauchemin_arthorBBs_bromo-keton": "beauchemin_bromo_keton",
            "Beauchemin_arthorBBs_amines": "beauchemin_amines",
            "Lundgren_arthorBBs_aromatic_carboxylates": "lundgren_aromatic_carboxylates",
            "Lundgren_arthorBBs_alkenes_alcohol-and-ester": "lundgren_alkenes_alcohol_and_ester",
            "Lundgren_arthorBBs_alkynes": "lundgren_alkynes",
            "Lundgren_arthorBBs_alkynes_ester-or-ketone": "lundgren_alkynes_ester_or_ketone",
            "Lundgren_arthorBBs_aryl-boronic-acid": "lundgren_aryl_boronic_acid",
            "c1_arthorBBs_ra": "c1_ra",
            "c1_arthorBBs_rb": "c1_rb",
            "c2_ra": "c2_ra",
            "c4_ra": "c4_ra",
            "c4_rb": "c4_rb",
            "c5_a":  "c5_a",
            "c5_z":  "c5_z",
            "c5_b0": "c5_b0",
            "c5_b1": "c5_b1",
            "c5_b2": "c5_b2",
            "c6_az": "c6_az",
            "c6_b":  "c6_b",
            "c8_az": "c8_az",
            "c8_b":  "c8_b",
            "c8_c":  "c8_c",
            "c8_d":  "c8_d",
        }

    def search_similarity(self, component_name, query_smiles, sim_thresh, limit):
        """
        ULTRA-SIMPLE & FAST: Direct table query with KNN operator.
        No JOINs. No mapping tables. Just pure speed.
        """
        table_name = self.table_map.get(component_name)
        if not table_name:
            print(f"  Error: No table found for {component_name}")
            return []

        query = f"""
            SELECT zinc_id, smiles,
                   tanimoto_sml(mfp, morganbv_fp(%s::mol)) as similarity
            FROM {table_name}
            ORDER BY mfp <%%> morganbv_fp(%s::mol)
            LIMIT %s;
        """

        results = []
        try:
            with psycopg2.connect(**self.conn_params) as conn:
                with conn.cursor() as cursor:
                    cursor.execute(query, [query_smiles, query_smiles, limit])
                    rows = cursor.fetchall()
                    for zinc_id, smiles, similarity in rows:
                        if float(similarity) >= sim_thresh:
                            mol = Chem.MolFromSmiles(smiles)
                            if mol:
                                results.append({
                                    'id': zinc_id or "DB_MOL",
                                    'smiles': smiles,
                                    'mol': mol,
                                    'similarity': float(similarity)
                                })
            return results
        except Exception as e:
            print(f"  Database error: {e}")
            return []


REACTION_INFO = {
    "Rousseaux": {
        "a1": {
            "type": "conditional",
            "smarts": {
                "a1_1": "[OH]-[CH2:4]-[CH2:2]-[#6:1].[NX3H1:6]-[#6:7]>>[CD2]1-[CD3:2]([#6:1])-[CX4:4]1-[NX3;!$(NC=O):6]-[#6:7]",
                "a1_2": "[Br]-[CX4:4]-[CX4:2]-[#6:1].[CX4]-O-[CX3:3](=O)-[CX4:5].[NX3H1:6]-[#6:7]>>[CD2:3]1-[CD3:2]([#6:1])-[CX4:4]1([CX4:5])-[NX3;!$(NC=O):6]-[#6:7]"
            },
            "components": {
                "a1_1": ["Rousseaux_arthorBBs_OH1", "Rousseaux_arthorBBs_amines"],
                "a1_2": ["Rousseaux_arthorBBs_Br1", "Rousseaux_arthorBBs_C001", "Rousseaux_arthorBBs_amines"]
            }
        }
    },
    "Le": {
        "a2": {
            "type": "single",
            "smarts": "[#6:1][NH:2][#6:3]>>[#6:1][N:2]([#6:3])[C](=[O])[F]",
            "components": ["Le_arthorBBs_amines"]
        }
    },
    "Beauchemin": {
        "a3": {
            "type": "two_reagents",
            "smarts": "[#6:3]-[CX2H0:2]#[CX2H0:1]-[CX4H2:7]-[#6:8].[#6:9]-[NH2:10]>>[ND3:10]1(-[#6:9])-[CD3H0:1](=[CD2H1:7]-[#6:8])-[CD3:2](-[#6:3])=[ND2]-[ND2H1]-[CD3]-1=[O]",
            "components": ["Beauchemin_arthorBBs_alkynes", "Beauchemin_arthorBBs_amines"]
        },
        "a4": {
            "type": "dual_smarts",
            "smarts": {
                "hydrazine": "O=[C:2]([#6:3])-[CH2;!R:1]-Br.[N:9]-[NH2:10]>>[ND3:10]1(-[N:9])-[CD2H2:1]-[CD3:2](-[#6:3])=[ND2]-[ND2H1]-[CD3]-1=[O]",
                "primary_amine": "O=[C:2]([#6:3])-[CH2;!R:1]-Br.[#6:9]-[NH2:10]>>[ND3:10]1(-[#6:9])-[CD2H2:1]-[CD3:2](-[#6:3])=[ND2]-[ND2H1]-[CD3]-1=[O]"
            },
            "components": ["Beauchemin_arthorBBs_bromo-keton", "Beauchemin_arthorBBs_amines"]
        }
    },
    "Lundgren": {
        "a5": {
            "type": "dual_smarts",
            "smarts": {
                "alcohol": "[a:1][CH2:2]C(=O)[OH].[#6:4][CX3H:5]=[CX3H:6][CX4H2:7][OH]>>[a:1][CX4H2:2][CX4H1:5]([#6:4])[CX3H1:6]=[CX3H2:7]",
                "ester": "[a:1][CH2:2]C(=O)[OH].[#6:4][CX3H:5]=[CX3H:6][CX3:7](=O)[O][#6]>>[a:1][CX4H2:2][CX4H1:5]([#6:4])[CX3H1:6]=[CX3H2:7]"
            },
            "components": ["Lundgren_arthorBBs_aromatic_carboxylates", "Lundgren_arthorBBs_alkenes_alcohol-and-ester"]
        },
        "a6": {
            "type": "single",
            "smarts": "[a:1][CH2:2]C(=O)[OH]>>[a:1][CH2:2][C]1C=CCCC1",
            "components": ["Lundgren_arthorBBs_aromatic_carboxylates"]
        },
        "a7": {
            "type": "two_reagents",
            "smarts": "[*:1][CX2H0:2]#[CX2H1:3].[CX2H1:4]#[CX2H0:5][CX3H0:6](=[O:7])[*:8]>>[*:1][CX3H1:2]=[CH:3][CH:4]=[CH:5][CX3H0:6](=[O:7])[*:8]",
            "components": ["Lundgren_arthorBBs_alkynes", "Lundgren_arthorBBs_alkynes_ester-or-ketone"]
        },
        "a8": {
            "type": "three_reagents",
            "smarts": "[*:1][CX2H0:2]#[CX2H1:3].[CX2H1:4]#[CX2H0:5][CX3H0:6](=[O:7])[*:8].[c:11][B]([OH])[OH]>>[*:1][CX3H1:2]=[CH:3][CH:4]([c:11])[CH2:5][CX3H0:6](=[O:7])[*:8]",
            "components": ["Lundgren_arthorBBs_alkynes", "Lundgren_arthorBBs_alkynes_ester-or-ketone", "Lundgren_arthorBBs_aryl-boronic-acid"]
        }
    },
    "Batey": {
        "c1": {
            "type": "two_reagents",
            "smarts": "[O]-[C:1](=[O:2])-[C:3]-[C:4]=[O:6].[C:8](=[O:9])-[NH1:10]>>[C:8](=[O:9])-[N:10]-[C:1](=[O:2])-[C:3]-[C:4]=[O:6]",
            "components": ["c1_arthorBBs_ra", "c1_arthorBBs_rb"]
        },
        "c2": {
            "type": "single",
            "smarts": "[NX3;H2,H1;!$(NC=O):1]>>[N:1]c1nnns1",
            "components": ["c2_ra"]
        },
        "c4": {
            "type": "two_reagents",
            "smarts": "[NX3;H2,H1;!$(NC=O):1].S=C=N[*;!H;!$(C=O):3]>>[*:1](c1nnnn1[*:3])",
            "components": ["c4_ra", "c4_rb"]
        },
    },
    "Wood": {
        "c5": {
            "type": "conditional",
            "smarts": {
                # 6 generalized SMARTS: 3 chain lengths × 2 reagent-A types.
                # The "[*:8]" slot represents the EWG, which must be present on
                # the central carbon (missing from the published SMARTS).
                # Chiral / non-chiral are both matched by [C;X4;!R:5].
                # Variants ordered most-specific first (longest chain first,
                # sulfonamide before OH/NH/SH).
                "z_b2_gen": "[a:1][$([SX4](=[OX1])(=[OX1])[NX3H2])].[F,Cl,Br,I][C:3](=O)[CX4H2:7][CX4H2:6][C;X4;!R:5]([*:8])>>[C;X4;!R:5]([a:1])([*:8])[CX4H2:6][CX4H2:7][C:3](=O)O",
                "a_b2_gen": "[a:1][OX2H1,NX3H2,SX2H1:2].[F,Cl,Br,I][C:3](=O)[CX4H2:7][CX4H2:6][C;X4;!R:5]([*:8])>>[C;X4;!R:5]([a:1])([*:8])[CX4H2:6][CX4H2:7][C:3](=O)[*:2]",
                "z_b1_gen": "[a:1][$([SX4](=[OX1])(=[OX1])[NX3H2])].[F,Cl,Br,I][C:3](=O)[CX4H2:6][C;X4;!R:5]([*:8])>>[C;X4;!R:5]([a:1])([*:8])[CX4H2:6][C:3](=O)O",
                "a_b1_gen": "[a:1][OX2H1,NX3H2,SX2H1:2].[F,Cl,Br,I][C:3](=O)[CX4H2:6][C;X4;!R:5]([*:8])>>[C;X4;!R:5]([a:1])([*:8])[CX4H2:6][C:3](=O)[*:2]",
                "z_b0_gen": "[a:1][$([SX4](=[OX1])(=[OX1])[NX3H2])].[F,Cl,Br,I][C:3](=O)[C;X4;!R:5]([*:8])>>[C;X4;!R:5]([a:1])([*:8])[C:3](=O)O",
                "a_b0_gen": "[a:1][OX2H1,NX3H2,SX2H1:2].[F,Cl,Br,I][C:3](=O)[C;X4;!R:5]([*:8])>>[C;X4;!R:5]([a:1])([*:8])[C:3](=O)[*:2]",
            },
            "components": {
                "z_b2_gen": ["c5_z", "c5_b2"],
                "a_b2_gen": ["c5_a", "c5_b2"],
                "z_b1_gen": ["c5_z", "c5_b1"],
                "a_b1_gen": ["c5_a", "c5_b1"],
                "z_b0_gen": ["c5_z", "c5_b0"],
                "a_b0_gen": ["c5_a", "c5_b0"],
            }
        }
    },
    "West": {
        # c6 has 2 variants: methacrylate (R2 = R group, non-H) and acrylate
        # (R2 = H).  Both search the merged c6_az (a + z merged: chemistry
        # is convergent, can't distinguish from product) and c6_b (acrylates
        # + methacrylates merged).
        # Ordered most-specific first: methacrylate before acrylate, since the
        # methacrylate pattern is stricter and won't false-match acrylate hits.
        "c6": {
            "type": "conditional",
            "smarts": {
                "c6_meth": "C(=O)(OC=O)[*:1].[CX3H2]=C([*:2])([*:3])>>C1(=C2C(CCC1)C(C2)([*:2])[*:3])OC([*:1])=O",
                "c6_acry": "C(=O)(OC=O)[*:1].[CX3H2]=[CX3H1][*:2]>>C1(=C2C(CCC1)C(C2)[*:2])OC([*:1])=O",
            },
            "components": {
                "c6_meth": ["c6_az", "c6_b"],
                "c6_acry": ["c6_az", "c6_b"],
            }
        },
        # c8 has 3 variants: 3 ring chemistries (furan / cyclopentadiene /
        # pyrrole).  All search c8_az for reagent A and the appropriate
        # ring-specific table for reagent B.
        "c8": {
            "type": "conditional",
            "smarts": {
                "c8_b_furan":     "C(=O)(OC=O)[*:1].[c:2]1[o:3][c:4][c:5][c:6]1>>C1(=C2C(CCC1)[C:2]3[O:3][C:4]2[C:5]=[C:6]3)OC(=O)[*:1]",
                "c8_c_cp":        "C(=O)(OC=O)[*:1].[C:2]=1[C:3][C:4]=[C:5][C:6]1>>C1(=C2C(CCC1)[C:2]3[C:3][C:4]2[C:5]=[C:6]3)OC(=O)[*:1]",
                "c8_d_pyrrole":   "C(=O)(OC=O)[*:1].[c:2]1[n:3][c:4][c:5][c:6]1>>C1(=C2C(CCC1)[C:2]3[N:3][C:4]2[C:5]=[C:6]3)OC(=O)[*:1]",
            },
            "components": {
                "c8_b_furan":   ["c8_az", "c8_b"],
                "c8_c_cp":      ["c8_az", "c8_c"],
                "c8_d_pyrrole": ["c8_az", "c8_d"],
            }
        }
    },

}


class HitExpansionPipeline:
    def __init__(self, db: PCCLDatabase):
        self.db = db
        self.morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

    def decompose_hit(self, hit_smiles, lab, reaction):
        rxn_info = REACTION_INFO[lab][reaction]
        r_type = rxn_info["type"]
        hit_mol = Chem.MolFromSmiles(hit_smiles)

        if not hit_mol: return [], [], None

        selected_smarts = None
        target_components = []
        selected_variant = None

        # Substructure filters per BB table. After a variant decomposes, we
        # check that each decomposed reagent actually matches what its target
        # table contains (e.g. a c5_z reagent really must be an aryl sulfonamide).
        # This rejects "decompositions" that RDKit produces from chemically
        # implausible matches.
        SUBSTRUCTURE_FILTERS = {
            "c5_a": "[a][OX2H1,NX3H2,SX2H1]",            # Ar-OH/NH2/SH
            "c5_z": "[a][SX4](=[OX1])(=[OX1])[NX3H2]",   # Ar-SO2NH2
        }
        def _passes_filter(reagent_mol, comp_name):
            pat = SUBSTRUCTURE_FILTERS.get(comp_name)
            if pat is None:
                return True
            return reagent_mol.HasSubstructMatch(Chem.MolFromSmarts(pat))

        def _normalize_reagent(m):
            """Clear stale explicit-H counts on non-aromatic atoms so RDKit
            recomputes correct hydrogens. Needed because RunReactants in
            reverse mode often leaves heteroatoms (e.g. NH2 → NH) with wrong
            H counts. Aromatic atoms are left alone to preserve [nH] tautomers."""
            m2 = Chem.Mol(m)
            for atom in m2.GetAtoms():
                if not atom.GetIsAromatic():
                    atom.SetNumExplicitHs(0)
                    atom.SetNoImplicit(False)
            try:
                Chem.SanitizeMol(m2)
            except Exception:
                return None
            if m2.GetNumAtoms() == 0:
                return None
            return m2

        def _formate_to_chloride(m):
            """Convert formate-mixed-anhydride O=COC(=O)Ar → acyl chloride
            O=C(Cl)Ar. Used for c6/c8 only: the reverse SMARTS produces a
            formate-mixed-anhydride as the A reagent (because that's what the
            forward SMARTS wrote), but the real BBs in c6_az/c8_az are acyl
            chlorides or cyclic anhydrides, neither of which look anything
            like the formate form (Tanimoto ~0.26-0.63). Converting to acyl
            chloride form makes similarity search work properly (gives 1.000
            against the real chloride BBs)."""
            try:
                rxn = AllChem.ReactionFromSmarts(
                    "[CH1:1](=O)[O:2][C:3](=O)[*:4]>>[Cl][C:3](=O)[*:4]"
                )
                products = rxn.RunReactants((m,))
                if not products:
                    return m   # didn't match the pattern; return unchanged
                for ps in products:
                    if len(ps) >= 1:
                        result = ps[0]
                        try:
                            Chem.SanitizeMol(result)
                            return result
                        except Exception:
                            continue
                return m
            except Exception:
                return m

        if r_type == "conditional":
            # Iterate over variants in dict insertion order. The dict should be
            # ordered most-specific-first so the right variant wins.
            for variant_name, test_smarts in rxn_info["smarts"].items():
                parts = test_smarts.split('>>')
                test_reverse = AllChem.ReactionFromSmarts(f"{parts[1]}>>{parts[0]}")
                if test_reverse is None:
                    continue
                try:
                    all_products = test_reverse.RunReactants((hit_mol,))
                except:
                    continue
                if not all_products:
                    continue
                target_components_v = rxn_info["components"][variant_name]
                # Try every product set, not just the first -- RDKit often
                # returns several alternative decompositions, only some valid.
                accepted = False
                for prod_set in all_products:
                    # Pick the right post-processing.  For reactions whose
                    # published SMARTS leave decomposed reagents with stale
                    # explicit H counts (Wood c5), normalize.  For everything
                    # else (Rousseaux a1, etc.) the published SMARTS produce
                    # intentionally-quirky placeholders that get broken by
                    # H normalization, so just sanitize directly.
                    use_normalize = (reaction == "c5")
                    reagents = []
                    bad = False
                    for rmol in prod_set:
                        if use_normalize:
                            norm = _normalize_reagent(rmol)
                            if norm is None:
                                bad = True
                                break
                            reagents.append(norm)
                        else:
                            try:
                                Chem.SanitizeMol(rmol)
                                if rmol.GetNumAtoms() == 0:
                                    bad = True
                                    break
                                reagents.append(rmol)
                            except Exception:
                                bad = True
                                break
                    if bad:
                        continue
                    # c6/c8 specific: convert formate-mixed-anhydride A reagent
                    # to acyl chloride form so it matches the real BBs in
                    # c6_az / c8_az tables (which contain chlorides/anhydrides,
                    # NOT formate-mixed-anhydrides).
                    if reaction in ("c6", "c8") and reagents:
                        reagents[0] = _formate_to_chloride(reagents[0])
                    # Validate reagents against target tables' substructure filters
                    if all(_passes_filter(r, c)
                           for r, c in zip(reagents, target_components_v)):
                        selected_smarts = test_smarts
                        target_components = target_components_v
                        selected_variant = variant_name
                        accepted = True
                        break
                if accepted:
                    break
            if not selected_smarts:
                return [], [], None

        elif r_type == "dual_smarts" and reaction == "a4":
            hydrazine = Chem.MolFromSmarts("[NX3]-[NX3]")
            sv = "hydrazine" if (hydrazine and hit_mol.HasSubstructMatch(hydrazine)) else "primary_amine"
            selected_smarts = rxn_info["smarts"][sv]
            target_components = rxn_info["components"]
            selected_variant = sv

        elif r_type == "dual_smarts":
            for name, smarts in rxn_info["smarts"].items():
                parts = smarts.split('>>')
                test_reverse = AllChem.ReactionFromSmarts(f"{parts[1]}>>{parts[0]}")
                try:
                    products = test_reverse.RunReactants((hit_mol,))
                    if products and len(products) > 0:
                        selected_smarts = smarts
                        target_components = rxn_info["components"]
                        selected_variant = name
                        break
                except:
                    continue
            if not selected_smarts:
                selected_smarts = list(rxn_info["smarts"].values())[0]
                target_components = rxn_info["components"]

        else:
            selected_smarts = rxn_info["smarts"]
            target_components = rxn_info["components"]

        parts = selected_smarts.split('>>')
        reverse_rxn = AllChem.ReactionFromSmarts(f"{parts[1]}>>{parts[0]}")
        self.forward_reaction = AllChem.ReactionFromSmarts(selected_smarts)

        try:
            products = reverse_rxn.RunReactants((hit_mol,))
            if products and len(products) > 0:
                reagents = []
                for rmol in products[0]:
                    Chem.SanitizeMol(rmol)
                    reagents.append(rmol)
                return reagents, target_components, selected_variant
        except Exception as e:
            print(f"  Decomposition error: {e}")
        return [], [], None

    def calculate_druglikeness(self, mol):
        try:
            mw = Descriptors.MolWt(mol)
            logp = Crippen.MolLogP(mol)
            rotb = Lipinski.NumRotatableBonds(mol)
            psa = Descriptors.TPSA(mol)

            # Veber violations: Rotatable bonds > 10 OR PSA > 140
            veber_vios = (rotb > 10) + (psa > 140)

            return {
                'MW': round(mw, 3),
                'LogP': round(logp, 2),
                'HBD': Lipinski.NumHDonors(mol),
                'HBA': Lipinski.NumHAcceptors(mol),
                'ROTB': rotb,
                'PSA': round(psa, 2),
                'HAC': Lipinski.HeavyAtomCount(mol),
                'FSP3': round(Descriptors.FractionCSP3(mol), 2),
                'QED': round(QED.qed(mol), 3),
                'Lipinski_violations': sum([mw > 500, logp > 5, Lipinski.NumHDonors(mol) > 5, Lipinski.NumHAcceptors(mol) > 10]),
                'Veber_violations': veber_vios # <--- Add this
            }
        except: return None

    def run_pipeline(self, hit_smiles, lab, reaction, sim_thresh, limit, max_vios, max_prods):
        print(f"\n[1] Decomposing {hit_smiles} locally...")
        hit_mol = Chem.MolFromSmiles(hit_smiles)
        if not hit_mol:
            print("  Invalid SMILES!")
            return pd.DataFrame()

        original_fp = self.morgan_gen.GetFingerprint(hit_mol)
        reagents, components, variant = self.decompose_hit(hit_smiles, lab, reaction)

        if not reagents:
            print("  Decomposition failed!")
            return pd.DataFrame()

        print(f"  Decomposed into {len(reagents)} reagents:")
        for i, r in enumerate(reagents):
            print(f"    Reagent {i+1}: {Chem.MolToSmiles(r)}")

        similar_lists = []
        for i, (rmol, comp_name) in enumerate(zip(reagents, components)):
            t0 = time.time()
            print(f"\n[2] Searching for Reagent {i+1} in '{comp_name}'...")

            db_results = self.db.search_similarity(
                component_name=comp_name,
                query_smiles=Chem.MolToSmiles(rmol),
                sim_thresh=sim_thresh,
                limit=limit
            )
            elapsed = time.time() - t0
            print(f"  ✓ {len(db_results)} matches in {elapsed:.1f}s")
            if db_results:
                print(f"    Top: {db_results[0]['smiles']} (sim: {db_results[0]['similarity']:.3f})")
            similar_lists.append(db_results)

        for i, sl in enumerate(similar_lists):
            if len(sl) == 0:
                print(f"\n  ⚠ Reagent {i+1} returned 0 matches!")
                return pd.DataFrame()

        print(f"\n[3] Enumerating analogs...")
        products = []
        for i, combo in enumerate(itertools.product(*similar_lists)):
            if i >= max_prods: break
            try:
                prods = self.forward_reaction.RunReactants(tuple(r['mol'] for r in combo))
                if prods:
                    pmol = prods[0][0]
                    Chem.SanitizeMol(pmol)
                    props = self.calculate_druglikeness(pmol)

                    if props and props['Lipinski_violations'] <= max_vios:
                        product_fp = self.morgan_gen.GetFingerprint(pmol)
                        sim_to_hit = DataStructs.TanimotoSimilarity(original_fp, product_fp)

                        products.append({
                            'Smiles': Chem.MolToSmiles(pmol),
                            'Code': "_".join(r['id'] for r in combo),
                            'product_similarity_to_hit': round(sim_to_hit, 3),
                            **props
                        })
            except: continue

        df = pd.DataFrame(products)
        if not df.empty:
            df = df.drop_duplicates(subset=['Smiles']).sort_values('product_similarity_to_hit', ascending=False)
        return df

print("✓ Separate-Table Cloud Pipeline Ready!")


# ============================================================================
# Launch the GUI (continuation of the cell above)
# ============================================================================
# ============================================================================
# CELL 3: Dynamic User Interface & Execution
# ============================================================================
#@title User Interface & Execution
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

# 1. Define configurations
reaction_mapping = {
    "Rousseaux": ["a1"],
    "Le": ["a2"],
    "Beauchemin": ["a3", "a4"],
    "Lundgren": ["a5", "a6", "a7", "a8"],
    "Batey": ["c1", "c2", "c4"],
    "Wood": ["c5"],
    "West": ["c6", "c8"]
}

default_smiles = {
    "a1": "N#CCCCN1CCN(C2([C@H](Br)c3ccccc3)CC2CN2CCCCC2)CC1",
    "a2": "CC(C)N(C(=O)F)c1ccc(B2OC(C)(C)C(C)(C)O2)cc1",
    "a3": "O=C1NN=C2CC[C@@H]3[C@H](CO)[C@@H]3CC=C2N1c1cccc2cc(O)ccc12",
    "a4": "CN(C)/C=N/c1ccc(N2CC(c3cccc(O)c3)=NNC2=O)cc1C#N",
    "a5": "C=CC(Cc1ccc(OC)c(S(=O)(=O)N2CCOCC2)c1)c1ccc(C(F)(F)F)cc1",
    "a6": "CC(C)(C)OC(=O)NC(CCC(N)=O)C(=O)OCc1ccc(CC2C=CCCC2)cc1",
    "a7": "O=C(C=CC=Cc1ccc(Cl)cc1)OCc1ccccc1",
    "a8": "COC(=O)CC(C=CCn1cc(F)c(=O)[nH]c1=O)c1ccc(C(=O)NCCN2CCOCC2)cc1",
    "c1": "O=C(CC(=O)N1CCN(Cc2ccccc2)C1=O)Cc1cc(F)c(F)cc1F",
    "c2": "CCc1ccc(CCOc2ccc(CC3SC(Nc4nnns4)=NC3=O)cc2)nc1",
    "c4": "CCOC(=O)CCn1nnnc1NCc1nccs1",
    "c5": "CCOC(=O)C(C(=O)O)c1ccc(C(=O)Nc2sc(-c3ccccc3)cc2C(N)=O)cc1",
    "c6": "COc1cc[c]c(C(=O)OC2=C3CC(C)(C(=O)OCCN4CCCCC4)C3CCC2)c1",
    "c8": "N[C@H](CC1=CC2OC1C1=C(OC(=O)c3cccc4cc(Br)c[c]c34)CCCC12)C(=O)O"
}

# 2. Create UI Widgets
lab_dropdown = widgets.Dropdown(options=list(reaction_mapping.keys()), value="Rousseaux", description='Lab:')
reaction_dropdown = widgets.Dropdown(options=reaction_mapping["Rousseaux"], value="a1", description='Reaction:')
hit_smiles_input = widgets.Text(value=default_smiles["a1"], description='Hit SMILES:', style={'description_width': 'initial'}, layout=widgets.Layout(width='600px'))

sim_thresh_slider = widgets.FloatSlider(value=0.35, min=0.1, max=1.0, step=0.05, description='Similarity Threshold:')
max_reagents_slider = widgets.IntSlider(value=10, min=5, max=100, step=5, description='Max Reagents:')
max_vios_slider = widgets.IntSlider(value=1, min=0, max=2, step=1, description='Max Violations:')
max_prods_slider = widgets.IntSlider(value=1000, min=1000, max=100000, step=1000, description='Max Products:')

run_button = widgets.Button(description="Run Pipeline", button_style='success')
output_area = widgets.Output()

# 3. Logic to update options and SMILES
def update_ui(change):
    if change['owner'] == lab_dropdown:
        new_lab = change['new']
        reaction_dropdown.options = reaction_mapping[new_lab]
        reaction_dropdown.value = reaction_mapping[new_lab][0]

    hit_smiles_input.value = default_smiles[reaction_dropdown.value]

lab_dropdown.observe(update_ui, names='value')
reaction_dropdown.observe(update_ui, names='value')

# 4. Define Execution Logic
def on_button_clicked(b):
    with output_area:
        clear_output()
        print(f"Starting pipeline for {lab_dropdown.value} / {reaction_dropdown.value}...")
        start_time = time.time()

        # --- INITIALIZATION BLOCK ---
        # Ensure the pipeline and db are initialized here
        db = PCCLDatabase()
        pipeline = HitExpansionPipeline(db)

        # Pipeline execution
        global results_df
        results_df = pipeline.run_pipeline(
            hit_smiles=hit_smiles_input.value,
            lab=lab_dropdown.value,
            reaction=reaction_dropdown.value,
            sim_thresh=sim_thresh_slider.value,
            limit=max_reagents_slider.value,
            max_vios=max_vios_slider.value,
            max_prods=max_prods_slider.value
        )

        if len(results_df) > 0:
            print(f"\n✓ SUCCESS! Generated {len(results_df)} analogs.")
            # Displaying without index
            display(results_df[['Smiles', 'Code', 'MW', 'LogP', 'product_similarity_to_hit']]
                    .head(10)
                    .style.hide(axis='index'))
        else:
            print("\n✗ No analogs generated. Try lowering the similarity threshold.")
        print(f"\nWall time: {time.time() - start_time:.2f} seconds")

run_button.on_click(on_button_clicked)

# 5. Display the UI
display(lab_dropdown, reaction_dropdown, hit_smiles_input, sim_thresh_slider,
        max_reagents_slider, max_vios_slider, max_prods_slider, run_button, output_area)

✓ Separate-Table Cloud Pipeline Ready!


Dropdown(description='Lab:', options=('Rousseaux', 'Le', 'Beauchemin', 'Lundgren', 'Batey', 'Wood'), value='Ro…

Dropdown(description='Reaction:', options=('a1',), value='a1')

Text(value='N#CCCCN1CCN(C2([C@H](Br)c3ccccc3)CC2CN2CCCCC2)CC1', description='Hit SMILES:', layout=Layout(width…

FloatSlider(value=0.35, description='Similarity Threshold:', max=1.0, min=0.1, step=0.05)

IntSlider(value=10, description='Max Reagents:', min=5, step=5)

IntSlider(value=1, description='Max Violations:', max=2)

IntSlider(value=1000, description='Max Products:', max=100000, min=1000, step=1000)

Button(button_style='success', description='Run Pipeline', style=ButtonStyle())

Output()

## After running a search
Run the cells below to analyze, visualize, and download the results.

In [ ]:
# ============================================================================
# CELL 6: Analyze and Visualize Results
# ============================================================================
#@title Analyze and Visualize Top Results
#@markdown Display top analogs

top_n = 10  #@param {type:"slider", min:2, max:50, step:2}

if len(results_df) == 0:
    print("No products generated to analyze!")
else:
    print(f"\n{'='*80}")
    print(f"TOP {top_n} ANALOGS BY SIMILARITY TO HIT")
    print(f"{'='*80}")

    # Sort by similarity to hit
    sorted_df = results_df.sort_values('product_similarity_to_hit', ascending=False).head(top_n)

    for i, (idx, row) in enumerate(sorted_df.iterrows(), 1):
        print(f"\n{i}. Code: {row['Code']}")
        print(f"   Similarity to hit: {row['product_similarity_to_hit']:.3f}")
        print(f"   SMILES: {row['Smiles']}")
        print(f"   MW: {row['MW']:.1f} | LogP: {row['LogP']:.2f} | HBD: {int(row['HBD'])} | HBA: {int(row['HBA'])}")
        print(f"   ROTB: {int(row['ROTB'])} | PSA: {row['PSA']:.1f} | FSP3: {row['FSP3']:.2f}")
        qed_value = f"{row['QED']:.3f}" if row['QED'] is not None else "N/A"
        print(f"   QED: {qed_value} | Total violations: {int(row['Lipinski_violations'] + row['Veber_violations'])}")
        if 'reagent1_id' in row:
            reagents = []
            for j in range(1, 4):
                reagent_key = f'reagent{j}_id'
                if reagent_key in row and pd.notna(row[reagent_key]):
                    reagents.append(row[reagent_key])
            print(f"   Reagents: {', '.join(reagents)}")

    # Display summary statistics
    print(f"\n{'='*80}")
    print("SUMMARY STATISTICS")
    print(f"{'='*80}")
    print(f"Total analogs generated: {len(results_df)}")

    if len(results_df) > 1:
        print(f"\nMolecular properties:")
        print(f"  Molecular Weight: {results_df['MW'].mean():.1f} ± {results_df['MW'].std():.1f} Da")
        print(f"  LogP: {results_df['LogP'].mean():.2f} ± {results_df['LogP'].std():.2f}")
        print(f"  HBA: {results_df['HBA'].mean():.1f} ± {results_df['HBA'].std():.1f}")
        print(f"  HBD: {results_df['HBD'].mean():.1f} ± {results_df['HBD'].std():.1f}")
        print(f"  ROTB: {results_df['ROTB'].mean():.1f} ± {results_df['ROTB'].std():.1f}")
        print(f"  PSA: {results_df['PSA'].mean():.1f} ± {results_df['PSA'].std():.1f}")
        print(f"  FSP3: {results_df['FSP3'].mean():.3f} ± {results_df['FSP3'].std():.3f}")

        # Drug-likeness statistics
        lipinski_compliant = len(results_df[results_df['Lipinski_violations'] == 0])
        veber_compliant = len(results_df[results_df['Veber_violations'] == 0])
        fully_compliant = len(results_df[(results_df['Lipinski_violations'] == 0) & (results_df['Veber_violations'] == 0)])

        print(f"\nDrug-likeness compliance:")
        print(f"  Lipinski compliant (0 violations): {lipinski_compliant} ({lipinski_compliant/len(results_df)*100:.1f}%)")
        print(f"  Veber compliant (0 violations): {veber_compliant} ({veber_compliant/len(results_df)*100:.1f}%)")
        print(f"  Fully compliant (0 violations for both): {fully_compliant} ({fully_compliant/len(results_df)*100:.1f}%)")

        # Similarity distribution
        print(f"\nSimilarity distribution:")
        print(f"  Mean similarity to hit: {results_df['product_similarity_to_hit'].mean():.3f}")
        print(f"  Max similarity: {results_df['product_similarity_to_hit'].max():.3f}")
        print(f"  Min similarity: {results_df['product_similarity_to_hit'].min():.3f}")

In [ ]:
# ============================================================================
# CELL 7: Visualize Top Molecules
# ============================================================================
#@title Visualize Top Molecules (Images)
#@markdown Show molecular structures
from rdkit.Chem import Draw
n_display = 8  #@param {type:"slider", min:4, max:24, step:4}

if len(results_df) > 0:
    top_mols = []
    top_legends = []

    for i, (idx, row) in enumerate(results_df.head(n_display).iterrows(), 1):
        mol = Chem.MolFromSmiles(row['Smiles'])
        if mol:
            top_mols.append(mol)
            legend = f"#{i} Sim: {row['product_similarity_to_hit']:.3f}\nMW: {row['MW']:.0f} LogP: {row['LogP']:.1f}"
            top_legends.append(legend)

    img = Draw.MolsToGridImage(top_mols, molsPerRow=4, subImgSize=(300, 300),
                               legends=top_legends, returnPNG=False)
    display(img)

In [ ]:
# ============================================================================
# CELL 8: Save and Download Results
# ============================================================================
#@title Save and Download Results

if len(results_df) > 0:
    # Save full results
    output_columns = ['Smiles', 'Code', 'MW', 'HAC', 'LogP', 'HBA', 'HBD',
                     'ROTB', 'FSP3', 'PSA', 'QED', 'product_similarity_to_hit']
    output_df = results_df[output_columns].copy()

    # Use 'lab' and 'reaction' variables defined in Cell 3
    output_file = f'hit_expansion_{lab}_{reaction}_results.csv'

    output_df.to_csv(output_file, index=False)
    print(f"✓ Results saved to: {output_file}")

    # Download file
    from google.colab import files
    files.download(output_file)
    print(f"✓ File downloaded!")
else:
    print("No results to save!")